# Chicago FY 2026 Budget — Exploratory Data Analysis

This notebook walks through the City of Chicago's **2026 budget ordinance** data step by step:

1. **Setup** — imports, configuration, helper functions  
2. **Data acquisition** — fetch from the [Chicago Data Portal](https://data.cityofchicago.org/) or load cached files  
3. **First look** — schema, row counts, sample records  
4. **Data quality** — missing values, duplicate line items, sanity checks  
5. **Appropriations** — spending by fund, department, and account  
6. **Council changes** — where the adopted budget differs from the Mayor's recommendation  
7. **Revenues** — estimated income by fund and revenue type  
8. **Summary** — headline numbers and next steps for the dashboard

> **Tip:** Run cells top-to-bottom on first open. Set `REFRESH_DATA = False` (default) to analyze the cached files in `output/` without hitting the API.

---
## 1. Setup

In [9]:
# Uncomment on first run if packages are missing
# %pip install sodapy pandas python-dotenv pyarrow plotly

from __future__ import annotations

import os
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.io as pio
from dotenv import load_dotenv
from sodapy import Socrata

load_dotenv()

ROOT = Path("..").resolve()  # src/
OUTPUT = ROOT / "data"
OUTPUT.mkdir(exist_ok=True)

APPRO_PATH = OUTPUT / "budget_appropriations_comparison_2026.csv"
REV_PATH = OUTPUT / "budget_revenues_2026.csv"

# Set True to re-download from the API; False loads cached CSVs (faster)
REFRESH_DATA = False

pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
pio.templates.default = "plotly_white"

print(f"Project root: {ROOT}")
print(f"Refresh from API: {REFRESH_DATA}")

Project root: /Users/dvgenis/Desktop/Big_Projects/Chicago_Budget_Dashboard
Refresh from API: False


### 1.1 Configuration & helper functions

Three Socrata datasets feed this analysis:

| Dataset | ID | Description |
|---|---|---|
| Ordinance appropriations | `6694-f78c` | Adopted spending line items |
| Recommended appropriations | `axxr-vais` | Mayor's recommended spending |
| Ordinance revenues | `nydj-5nax` | Estimated revenue by source |

In [10]:
DOMAIN = "data.cityofchicago.org"
APP_TOKEN = os.getenv("SOCRATA_APP_TOKEN")
USERNAME = os.getenv("SOCRATA_USERNAME")
PASSWORD = os.getenv("SOCRATA_PASSWORD")

DATASETS = {
    "ordinance_appropriations_2026": "6694-f78c",
    "recommended_appropriations_2026": "axxr-vais",
    "ordinance_revenue_2026": "nydj-5nax",
}

MERGE_KEYS = ["fund_code", "department_code", "account_code"]
MONEY_COLS = ["ordinance_amount", "recommended_amount", "delta_amount", "estimated_revenue"]


def get_client() -> Socrata:
    return Socrata(DOMAIN, app_token=APP_TOKEN, username=USERNAME, password=PASSWORD, timeout=60)


def fetch_all_records(client: Socrata, dataset_id: str, limit: int = 50_000) -> pd.DataFrame:
    offset = 0
    records: list[dict] = []
    while True:
        batch = client.get(dataset_id, limit=limit, offset=offset, order=":id")
        if not batch:
            break
        records.extend(batch)
        offset += limit
        print(f"  … {len(records):,} rows fetched")
    return pd.DataFrame.from_records(records)


def _parse_money(series: pd.Series) -> pd.Series:
    return (
        series.astype(str)
        .str.replace(r"[\$,]", "", regex=True)
        .pipe(pd.to_numeric, errors="coerce")
        .fillna(0.0)
    )


def _strip_strings(df: pd.DataFrame) -> pd.DataFrame:
    text_cols = df.select_dtypes(include=["object", "string"]).columns
    if len(text_cols):
        df = df.copy()
        df[text_cols] = df[text_cols].apply(lambda s: s.str.strip())
    return df


def clean_appropriations(df: pd.DataFrame, *, is_recommendation: bool = False) -> pd.DataFrame:
    amt_candidates = [c for c in df.columns if any(k in c.lower() for k in ("amount", "ordinance", "recommendation"))]
    if not amt_candidates:
        raise ValueError(f"No amount column found in {list(df.columns)}")
    amt_col = amt_candidates[0]
    target_amt_col = "recommended_amount" if is_recommendation else "ordinance_amount"

    df = df.rename(
        columns={
            "fund_description": "fund_name",
            "department_number": "department_code",
            "department_description": "department_name",
            "appropriation_account": "account_code",
            "appropriation_account_description": "account_name",
            amt_col: target_amt_col,
        }
    )
    df[target_amt_col] = _parse_money(df[target_amt_col])
    df["department_code"] = pd.to_numeric(df["department_code"], errors="coerce").fillna(0).astype(int)
    df = _strip_strings(df)

    keep = [c for c in [*MERGE_KEYS, "fund_name", "department_name", "account_name", target_amt_col] if c in df.columns]
    return df[keep]


def clean_revenues(df: pd.DataFrame) -> pd.DataFrame:
    df = df.rename(columns={"fund_name": "fund_name"})
    df["estimated_revenue"] = _parse_money(df["estimated_revenue"])
    return _strip_strings(df)


def merge_appropriations(enacted: pd.DataFrame, recommended: pd.DataFrame) -> pd.DataFrame:
    merged = pd.merge(enacted, recommended, on=MERGE_KEYS, how="outer", suffixes=("_enacted", "_recommended"))
    for base in ("fund_name", "department_name", "account_name"):
        enacted_col, rec_col = f"{base}_enacted", f"{base}_recommended"
        if enacted_col in merged.columns and rec_col in merged.columns:
            merged[base] = merged[enacted_col].combine_first(merged[rec_col])
            merged.drop(columns=[enacted_col, rec_col], inplace=True)

    merged["recommended_amount"] = merged["recommended_amount"].fillna(0.0)
    merged["ordinance_amount"] = merged["ordinance_amount"].fillna(0.0)
    merged["delta_amount"] = merged["ordinance_amount"] - merged["recommended_amount"]
    merged["pct_change"] = (
        merged["delta_amount"] / merged["recommended_amount"].replace(0, pd.NA) * 100
    ).fillna(0.0)
    return merged


def spending_purpose(account_name: str) -> str:
    """Group account names into reader-friendly spending categories (same logic as app.py)."""
    name = str(account_name).lower()
    if "reserve" in name:
        return "Reserves"
    if any(w in name for w in ("bond", "interest on", "term note", "loan")):
        return "Debt"
    if any(w in name for w in ("salary", "wage", "overtime", "payroll", "fringe", "pension", "annuity", "trainee")):
        return "People and benefits"
    if "aldermanic expense" in name:
        return "Aldermanic expenses"
    if any(w in name for w in ("judgment", "tort", "outside counsel")):
        return "Lawsuits and claims"
    if any(w in name for w in ("construction of", "purchase of vehicles", "machinery", "vehicles")):
        return "Buildings, vehicles, and equipment"
    if any(w in name for w in ("delegate agenc", "homeless", "youth", "violence", "workforce", "rebate")):
        return "Help for residents"
    if any(w in name for w in ("professional", "technical service", "software", "maintenance", "contract")):
        return "Contracts and outside services"
    return "Other city costs"


def fmt_millions(value: float) -> str:
    return f"${value / 1_000_000:,.1f}M"


def fmt_billions(value: float) -> str:
    return f"${value / 1_000_000_000:,.2f}B"

---
## 2. Data acquisition

Pull fresh data from Socrata **or** read the processed files already in `output/`.

In [11]:
def build_datasets(refresh: bool = REFRESH_DATA) -> tuple[pd.DataFrame, pd.DataFrame]:
    if refresh:
        client = get_client()
        try:
            print("Fetching ordinance appropriations…")
            raw_enacted = fetch_all_records(client, DATASETS["ordinance_appropriations_2026"])
            print("Fetching recommended appropriations…")
            raw_rec = fetch_all_records(client, DATASETS["recommended_appropriations_2026"])
            print("Fetching ordinance revenues…")
            raw_rev = fetch_all_records(client, DATASETS["ordinance_revenue_2026"])
        finally:
            client.close()

        clean_enacted = clean_appropriations(raw_enacted, is_recommendation=False)
        clean_rec = clean_appropriations(raw_rec, is_recommendation=True)
        appropriations = merge_appropriations(clean_enacted, clean_rec)
        revenues = clean_revenues(raw_rev)

        appropriations.to_csv(APPRO_PATH, index=False)
        appropriations.to_parquet(APPRO_PATH.with_suffix(".parquet"), index=False)
        revenues.to_csv(REV_PATH, index=False)
        revenues.to_parquet(REV_PATH.with_suffix(".parquet"), index=False)
        print(f"Saved → {APPRO_PATH.name}, {REV_PATH.name}")
    else:
        if not APPRO_PATH.exists() or not REV_PATH.exists():
            raise FileNotFoundError(
                f"Missing cached files in {OUTPUT}. Set REFRESH_DATA = True to download."
            )
        appropriations = pd.read_csv(APPRO_PATH)
        revenues = pd.read_csv(REV_PATH)
        print(f"Loaded cached files from {OUTPUT}/")

    for col in MONEY_COLS:
        if col in appropriations.columns:
            appropriations[col] = pd.to_numeric(appropriations[col], errors="coerce").fillna(0.0)
        if col in revenues.columns:
            revenues[col] = pd.to_numeric(revenues[col], errors="coerce").fillna(0.0)

    return appropriations, revenues


appropriations_raw, revenues = build_datasets()
print(f"Appropriations rows: {len(appropriations_raw):,}")
print(f"Revenue rows:        {len(revenues):,}")

Loaded cached files from /Users/dvgenis/Desktop/Big_Projects/Chicago_Budget_Dashboard/output/
Appropriations rows: 14,881
Revenue rows:        156


---
## 3. First look at the data

In [12]:
print("=== Appropriations (recommended vs adopted) ===")
display(appropriations_raw.head(8))
print("\n=== Revenues ===")
display(revenues.head(8))

=== Appropriations (recommended vs adopted) ===


,fund_code,department_code,account_code,ordinance_amount,recommended_amount,fund_name,department_name,account_name,delta_amount,pct_change
0,0075,5,0005,"1,235,614.00","1,235,614.00",Grants Management Fund,Office of Budget and Management,Salaries and Wages - on Payroll,0.00,0.00
1,0075,5,0015,"11,083.00","11,083.00",Grants Management Fund,Office of Budget and Management,Schedule Salary Adjustments,0.00,0.00
2,0075,5,0039,"26,108.00","26,108.00",Grants Management Fund,Office of Budget and Management,For the Employment of Students as Trainees,0.00,0.00
3,0075,5,0044,"723,000.00","723,000.00",Grants Management Fund,Office of Budget and Management,Fringe Benefits,0.00,0.00
4,0075,5,0140,"696,301.00","696,301.00",Grants Management Fund,Office of Budget and Management,For Professional and Technical Services and Ot...,0.00,0.00
5,0075,5,0152,0.00,0.00,Grants Management Fund,Office of Budget and Management,ADVERTISING,0.00,0.00
6,0075,5,0166,894.00,894.00,Grants Management Fund,Office of Budget and Management,"Dues, Subscriptions and Memberships",0.00,0.00
7,0075,5,0169,0.00,0.00,Grants Management Fund,Office of Budget and Management,TECHNICAL MEETING COSTS,0.00,0.00



=== Revenues ===


,fund_code,fund_name,revenue_group_type,revenue_category,revenue_source,estimated_revenue
0,0100,Corporate Fund,Intergovernmental Revenue,Municipal Auto Rental Tax,Municipal Auto Rental Tax,4894443
1,0100,Corporate Fund,Intergovernmental Revenue,Personal Property Replacement Tax,Personal Property Replacement Tax,202567148
2,0100,Corporate Fund,Intergovernmental Revenue,Reimbursements for City Services,Reimbursements for City Services,1143491
3,0100,Corporate Fund,Intergovernmental Revenue,State Income Tax,State Income Tax,545129731
4,0100,Corporate Fund,Local Non-Tax Revenue,"Leases, Rentals and Sales",Advertising Revenue,29300000
5,0100,Corporate Fund,Local Non-Tax Revenue,"Licenses, Permits, and Certificates",Alcohol Dealers' License,17328661
6,0100,Corporate Fund,Local Non-Tax Revenue,"Licenses, Permits, and Certificates",Building Permits,32797526
7,0100,Corporate Fund,Local Non-Tax Revenue,"Licenses, Permits, and Certificates",Business License,33861933


In [13]:
summary = pd.DataFrame(
    {
        "Appropriations": appropriations_raw.dtypes.astype(str),
        "Revenues": revenues.dtypes.astype(str),
    }
)
print("Column dtypes")
display(summary)

print("\nAppropriations describe (dollars):")
display(appropriations_raw[["ordinance_amount", "recommended_amount", "delta_amount"]].describe())

Column dtypes


,Appropriations,Revenues
account_code,str,NaN
account_name,str,NaN
delta_amount,float64,NaN
department_code,int64,NaN
department_name,str,NaN
estimated_revenue,NaN,int64
fund_code,str,str
fund_name,str,str
ordinance_amount,float64,NaN
pct_change,float64,NaN



Appropriations describe (dollars):


,ordinance_amount,recommended_amount,delta_amount
count,"14,881.00","14,881.00","14,881.00"
mean,"3,669,202.47","2,331,137.05","1,338,065.42"
std,"30,593,508.81","24,271,603.69","24,643,827.19"
min,"-56,600,000.00","-10,000,000.00","-450,646,229.00"
25%,"47,678.00","12,700.00",0.00
50%,"165,005.00","122,000.00",0.00
75%,"366,952.00","327,170.00","2,188.00"
max,"1,438,799,944.00","1,440,213,627.00","451,646,229.00"


---
## 4. Data quality checks

Before aggregating, we verify the data is complete and identify duplicate line items.

In [14]:
def quality_report(df: pd.DataFrame, keys: list[str], label: str) -> None:
    print(f"\n{'=' * 60}")
    print(label)
    print(f"{'=' * 60}")
    print(f"Rows:              {len(df):,}")
    print(f"Missing values:    {df.isna().sum().sum():,} total across all columns")
    dup_count = df.duplicated(keys, keep=False).sum()
    dup_groups = df.loc[df.duplicated(keys, keep=False), keys].drop_duplicates().shape[0]
    print(f"Duplicate keys:    {dup_count:,} rows in {dup_groups:,} key groups  ({', '.join(keys)})")

    if {"ordinance_amount", "recommended_amount"}.issubset(df.columns):
        zero_both = ((df["ordinance_amount"] == 0) & (df["recommended_amount"] == 0)).sum()
        print(f"Zero-dollar lines: {zero_both:,}")
    elif "estimated_revenue" in df.columns:
        zero_revenue = (df["estimated_revenue"] == 0).sum()
        print(f"Zero-revenue lines: {zero_revenue:,}")


quality_report(appropriations_raw, MERGE_KEYS, "Appropriations quality")
quality_report(revenues, ["fund_code", "revenue_source"], "Revenues quality")


Appropriations quality
Rows:              14,881
Missing values:    0 total across all columns
Duplicate keys:    12,611 rows in 357 key groups  (fund_code, department_code, account_code)
Zero-dollar lines: 223

Revenues quality
Rows:              156
Missing values:    202 total across all columns
Duplicate keys:    0 rows in 0 key groups  (fund_code, revenue_source)
Zero-revenue lines: 2


In [15]:
# Inspect a few duplicate key groups — same fund/dept/account with conflicting amounts
dup_mask = appropriations_raw.duplicated(MERGE_KEYS, keep=False)
dup_sample = (
    appropriations_raw.loc[dup_mask]
    .sort_values(MERGE_KEYS + ["ordinance_amount"])
    .head(12)
)
print("Sample duplicate line items (first 12 rows):")
display(
    dup_sample[
        ["fund_name", "department_name", "account_name", "recommended_amount", "ordinance_amount", "delta_amount"]
    ]
)

Sample duplicate line items (first 12 rows):


,fund_name,department_name,account_name,recommended_amount,ordinance_amount,delta_amount
10,Grants Management Fund,Department of Housing,Salaries and Wages - on Payroll,"404,374.00","404,374.00",0.00
11,Grants Management Fund,Department of Housing,Salaries and Wages - on Payroll,"448,928.00","404,374.00","-44,554.00"
12,Grants Management Fund,Department of Housing,Salaries and Wages - on Payroll,"404,374.00","448,928.00","44,554.00"
13,Grants Management Fund,Department of Housing,Salaries and Wages - on Payroll,"448,928.00","448,928.00",0.00
17,Grants Management Fund,Department of Housing,Schedule Salary Adjustments,"15,086.00","7,961.00","-7,125.00"
18,Grants Management Fund,Department of Housing,Schedule Salary Adjustments,"7,961.00","7,961.00",0.00
15,Grants Management Fund,Department of Housing,Schedule Salary Adjustments,"15,086.00","15,086.00",0.00
16,Grants Management Fund,Department of Housing,Schedule Salary Adjustments,"7,961.00","15,086.00","7,125.00"
19,Grants Management Fund,Department of Housing,Fringe Benefits,"232,540.00","232,540.00",0.00
20,Grants Management Fund,Department of Housing,Fringe Benefits,"268,111.00","232,540.00","-35,571.00"


In [16]:
# For analysis we keep the last record per merge key (most recent export row).
# Raw totals inflate ~3× when duplicates are included.
appropriations = appropriations_raw.drop_duplicates(MERGE_KEYS, keep="last").copy()
appropriations["purpose"] = appropriations["account_name"].map(spending_purpose)

print(f"Raw rows:     {len(appropriations_raw):,}")
print(f"Deduped rows: {len(appropriations):,}")
print(f"Raw ordinance total:     {fmt_billions(appropriations_raw['ordinance_amount'].sum())}")
print(f"Deduped ordinance total: {fmt_billions(appropriations['ordinance_amount'].sum())}")

Raw rows:     14,881
Deduped rows: 2,627
Raw ordinance total:     $54.60B
Deduped ordinance total: $15.60B


---
## 5. Appropriations overview

Headline spending totals and breakdowns by fund and department.

In [17]:
adopted_total = appropriations["ordinance_amount"].sum()
recommended_total = appropriations["recommended_amount"].sum()
council_change = adopted_total - recommended_total

headline = pd.DataFrame(
    {
        "Metric": [
            "Mayor's recommended appropriations",
            "Council-adopted appropriations",
            "Net council change",
            "Line items with any council change",
            "Unique funds",
            "Unique departments",
        ],
        "Value": [
            fmt_billions(recommended_total),
            fmt_billions(adopted_total),
            fmt_billions(council_change),
            f"{(appropriations['delta_amount'] != 0).sum():,}",
            f"{appropriations['fund_name'].nunique():,}",
            f"{appropriations['department_name'].nunique():,}",
        ],
    }
)
display(headline)

,Metric,Value
0,Mayor's recommended appropriations,$14.35B
1,Council-adopted appropriations,$15.60B
2,Net council change,$1.25B
3,Line items with any council change,146
4,Unique funds,47
5,Unique departments,40


In [18]:
by_fund = (
    appropriations.groupby("fund_name", as_index=False)
    .agg(adopted=("ordinance_amount", "sum"), recommended=("recommended_amount", "sum"))
    .assign(delta=lambda d: d["adopted"] - d["recommended"])
    .sort_values("adopted", ascending=False)
)

print("Top 10 funds by adopted appropriations:")
display(by_fund.head(10))

Top 10 funds by adopted appropriations:


,fund_name,adopted,recommended,delta
16,Corporate Fund,"5,881,407,231.00","5,690,675,951.00","190,731,280.00"
8,Chicago O'Hare Airport Fund,"2,057,945,114.00","2,050,384,004.00","7,561,110.00"
36,Policemen's Annuity and Benefit Fund,"1,145,063,924.00","1,106,262,515.00","38,801,409.00"
33,Municipal Employees' Annuity and Benefit Fund,"1,133,290,524.00","1,046,708,718.00","86,581,806.00"
20,Federal Grant Fund,"702,031,037.00","129,662,075.00","572,368,962.00"
45,Water Fund,"666,471,274.00","666,399,159.00","72,115.00"
21,Firemen's Annuity and Benefit Fund,"468,008,665.00","461,787,994.00","6,220,671.00"
39,Sewer Fund,"444,520,343.00","444,520,343.00",0.00
7,Chicago Midway Airport Fund,"431,075,029.00","429,493,892.00","1,581,137.00"
1,Bond Redemption and Interest Series Fund,"430,103,270.00","430,103,270.00",0.00


In [19]:
top_funds = by_fund.head(12).iloc[::-1]
fig = px.bar(
    top_funds,
    x="adopted",
    y="fund_name",
    orientation="h",
    title="Top 12 funds — adopted appropriations (FY 2026)",
    labels={"adopted": "Adopted ($)", "fund_name": ""},
    color_discrete_sequence=["#0c3b5e"],
)
fig.update_layout(height=520, yaxis_title="")
fig.show()

In [20]:
by_dept = (
    appropriations.groupby("department_name", as_index=False)
    .agg(adopted=("ordinance_amount", "sum"), recommended=("recommended_amount", "sum"))
    .assign(delta=lambda d: d["adopted"] - d["recommended"])
    .sort_values("adopted", ascending=False)
)

print("Top 10 departments by adopted appropriations:")
display(by_dept.head(10))

Top 10 departments by adopted appropriations:


,department_name,adopted,recommended,delta
31,Finance General,"8,190,935,451.00","8,053,719,737.00","137,215,714.00"
9,Chicago Police Department,"2,058,064,583.00","2,039,809,719.00","18,254,864.00"
4,Chicago Department of Aviation,"1,473,871,529.00","949,442,101.00","524,429,428.00"
7,Chicago Fire Department,"895,423,972.00","785,521,562.00","109,902,410.00"
6,Chicago Department of Transportation,"442,217,247.00","194,962,951.00","247,254,296.00"
22,Department of Fleet and Facility Management,"395,762,169.00","391,797,169.00","3,965,000.00"
20,Department of Family and Support Services,"379,683,511.00","296,157,674.00","83,525,837.00"
30,Department of Water Management,"262,147,797.00","263,396,197.00","-1,248,400.00"
5,Chicago Department of Public Health,"163,463,103.00","120,445,283.00","43,017,820.00"
26,Department of Planning and Development,"146,482,815.00","120,947,815.00","25,535,000.00"


In [21]:
by_purpose = (
    appropriations.groupby("purpose", as_index=False)["ordinance_amount"]
    .sum()
    .sort_values("ordinance_amount", ascending=False)
)

fig = px.pie(
    by_purpose,
    names="purpose",
    values="ordinance_amount",
    title="Adopted spending by purpose category",
    hole=0.35,
)
fig.update_traces(textposition="inside", textinfo="percent+label")
fig.update_layout(height=520)
fig.show()

---
## 6. Council changes (recommended → adopted)

Where did the City Council add or cut relative to the Mayor's budget?

In [22]:
changed = appropriations.loc[appropriations["delta_amount"] != 0].copy()
print(f"Line items changed by council: {len(changed):,}")
print(f"Total net increase: {fmt_billions(changed.loc[changed['delta_amount'] > 0, 'delta_amount'].sum())}")
print(f"Total net decrease: {fmt_billions(changed.loc[changed['delta_amount'] < 0, 'delta_amount'].sum())}")

Line items changed by council: 146
Total net increase: $1.52B
Total net decrease: $-0.28B


In [23]:
dept_changes = (
    changed.groupby("department_name", as_index=False)["delta_amount"]
    .sum()
    .sort_values("delta_amount", ascending=False)
)

print("Departments with largest council increases:")
display(dept_changes.head(8))
print("\nDepartments with largest council cuts:")
display(dept_changes.tail(8).sort_values("delta_amount"))

Departments with largest council increases:


,department_name,delta_amount
2,Chicago Department of Aviation,"524,429,428.00"
4,Chicago Department of Transportation,"247,254,296.00"
19,Finance General,"137,215,714.00"
5,Chicago Fire Department,"109,902,410.00"
12,Department of Family and Support Services,"83,525,837.00"
3,Chicago Department of Public Health,"43,017,820.00"
15,Department of Housing,"31,223,500.00"
16,Department of Planning and Development,"25,535,000.00"



Departments with largest council cuts:


,department_name,delta_amount
25,Office of the Mayor,"-3,935,004.00"
18,Department of Water Management,"-1,248,400.00"
13,Department of Finance,"-341,628.00"
24,Office of Public Safety Administration,"-78,607.00"
22,Office of City Clerk,"-52,376.00"
21,Office of Budget and Management,"-8,000.00"
9,Department of Buildings,-636.00
1,Chicago Commission on Human Relations,"8,000.00"


In [24]:
top_changes = dept_changes.reindex(dept_changes["delta_amount"].abs().nlargest(12).index).sort_values("delta_amount")
colors = ["#1c6b45" if v >= 0 else "#9c2f2f" for v in top_changes["delta_amount"]]

fig = px.bar(
    top_changes,
    x="delta_amount",
    y="department_name",
    orientation="h",
    title="Largest net council changes by department",
    labels={"delta_amount": "Council change ($)", "department_name": ""},
)
fig.update_traces(marker_color=colors)
fig.update_layout(height=520, yaxis_title="")
fig.show()

In [25]:
largest_line_changes = changed.reindex(changed["delta_amount"].abs().nlargest(15).index).sort_values("delta_amount")
display(
    largest_line_changes[
        ["fund_name", "department_name", "account_name", "recommended_amount", "ordinance_amount", "delta_amount", "pct_change"]
    ]
)

,fund_name,department_name,account_name,recommended_amount,ordinance_amount,delta_amount,pct_change
8009,Corporate Fund,Finance General,Scheduled Wage Adjustments,"360,500,000.00","262,287,301.00","-98,212,699.00",-27.24
10110,Community Safety Fund,Department of Family and Support Services,YOUTH EMPLOYMENT,"48,915,715.00",0.00,"-48,915,715.00",-100.00
8054,Corporate Fund,Finance General,Less Corporate Fund Savings,"-10,000,000.00","-56,600,000.00","-46,600,000.00",466.00
6461,Corporate Fund,Chicago Department of Public Health,Violence Reduction Program,0.00,"32,163,251.00","32,163,251.00",0.00
8064,Corporate Fund,Finance General,Policemen's Fund Advance Pension Payment,"32,738,939.00","70,717,328.00","37,978,389.00",116.00
9765,Policemen's Annuity and Benefit Fund,Finance General,For the City's Advance Contribution to Employe...,"33,448,415.00","72,249,824.00","38,801,409.00",116.00
10275,COVID-19 Grant Fund,Department of Family and Support Services,Reserve Balance,0.00,"39,285,159.00","39,285,159.00",0.00
6517,Corporate Fund,Department of Family and Support Services,Youth Employment,0.00,"48,915,715.00","48,915,715.00",0.00
14475,Local Public and Private Grant Fund,Chicago Department of Transportation,For Professional and Technical Services and Ot...,0.00,"58,864,000.00","58,864,000.00",0.00
8062,Corporate Fund,Finance General,Municipal Fund Advance Pension Payment,"59,639,679.00","128,823,928.00","69,184,249.00",116.00


---
## 7. Revenue analysis

In [26]:
revenue_total = revenues["estimated_revenue"].sum()

rev_headline = pd.DataFrame(
    {
        "Metric": ["Total estimated revenue", "Revenue line items", "Funds with revenue"],
        "Value": [
            fmt_billions(revenue_total),
            f"{len(revenues):,}",
            f"{revenues['fund_name'].nunique():,}",
        ],
    }
)
display(rev_headline)

,Metric,Value
0,Total estimated revenue,$14.62B
1,Revenue line items,156
2,Funds with revenue,36


In [27]:
by_group = (
    revenues.groupby("revenue_group_type", as_index=False)["estimated_revenue"]
    .sum()
    .sort_values("estimated_revenue", ascending=False)
)
by_group["share_pct"] = by_group["estimated_revenue"] / revenue_total * 100

display(by_group)

fig = px.bar(
    by_group,
    x="revenue_group_type",
    y="estimated_revenue",
    title="Estimated revenue by group type",
    labels={"estimated_revenue": "Estimated revenue ($)", "revenue_group_type": ""},
    color="revenue_group_type",
    color_discrete_map={
        "Local Tax": "#0c3b5e",
        "Local Non-Tax Revenue": "#3e6f8c",
        "Intergovernmental Revenue": "#a6843d",
        "Proceeds and Transfers In": "#8a4b32",
    },
)
fig.update_layout(showlegend=False, height=420)
fig.show()

,revenue_group_type,estimated_revenue,share_pct
2,Local Tax,3002058781,20.53
1,Local Non-Tax Revenue,1803689929,12.33
0,Intergovernmental Revenue,753734813,5.15
3,Proceeds and Transfers In,686698094,4.70


In [28]:
corporate = revenues.loc[revenues["fund_name"] == "Corporate Fund"].copy()
corp_by_cat = (
    corporate.groupby("revenue_category", as_index=False)["estimated_revenue"]
    .sum()
    .sort_values("estimated_revenue", ascending=False)
)

print(f"Corporate Fund revenue total: {fmt_billions(corporate['estimated_revenue'].sum())}")
display(corp_by_cat.head(10))

Corporate Fund revenue total: $6.25B


,revenue_category,estimated_revenue
17,Transaction Taxes,1393630329
13,Proceeds and Transfers In,686698094
16,State Income Tax,545129731
18,Transportation Taxes,497855406
3,"Fines, Forfeitures and Penalties",481667969
14,Recreation Taxes,425271489
1,Charges for Services,413619033
10,Municipal Public Utility Tax,395627804
5,Internal Service Earnings,357475666
11,Other Revenue,255834804


---
## 8. Spending vs. revenue (high level)

These datasets cover different scopes — appropriations span all city funds while revenue here is ordinance-level estimated income. Still useful as a sanity check on magnitude.

In [29]:
corp_spending = appropriations.loc[appropriations["fund_name"] == "Corporate Fund", "ordinance_amount"].sum()
corp_revenue = revenues.loc[revenues["fund_name"] == "Corporate Fund", "estimated_revenue"].sum()

compare = pd.DataFrame(
    {
        "Scope": ["All funds (deduped)", "Corporate Fund only"],
        "Adopted appropriations": [fmt_billions(adopted_total), fmt_billions(corp_spending)],
        "Estimated revenue": ["—", fmt_billions(corp_revenue)],
    }
)
display(compare)

if corp_revenue:
    gap = corp_spending - corp_revenue
    print(f"Corporate Fund gap (spending − revenue): {fmt_billions(gap)}")

,Scope,Adopted appropriations,Estimated revenue
0,All funds (deduped),$15.60B,—
1,Corporate Fund only,$5.88B,$6.25B


Corporate Fund gap (spending − revenue): $-0.36B


---
## 9. Key findings & next steps

**Data notes**
- The raw export contains duplicate `(fund_code, department_code, account_code)` keys; deduplicate before summing.
- ~150 line items differ between the Mayor's recommendation and the adopted ordinance.
- Largest fund by spending is the **Corporate Fund**; pension and airport funds also dominate totals.

**Outputs used by the dashboard**
- `output/budget_appropriations_comparison_2026.csv`
- `output/budget_revenues_2026.csv`

Run `python app.py` to launch the interactive Taipy dashboard built on these files.

In [30]:
# Optional: re-export cleaned deduped comparison for downstream use
EXPORT_DEDUPED = False

if EXPORT_DEDUPED:
    out_path = OUTPUT / "budget_appropriations_comparison_2026_deduped.csv"
    appropriations.to_csv(out_path, index=False)
    print(f"Wrote {out_path}")